# Existence equation — panel logit

Whether a cross-pool arbitrage exists. Here the **onset** risk set (`gap_lag == 0`): given no gap
is open, does one appear at `t`?

$$\Pr\!\big(D_{p,t}=1 \mid X_{p,t-1}\big)=\Lambda(\eta_{p,t}),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}$$

$$
\eta_{p,t}=\alpha_p
+\beta_2\,\log(\text{base\_fee}_t)+\beta_3\,\text{gas\_util}_{t-1}+\beta_4\,\log(1+\text{tip\_p90}_{t-1})
+\beta_5\,\overline{\log(1+\text{mev})}_{p,t-1}+\beta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}
+\beta_7\,\log(\text{ewma\_vol}_t)+\beta_8\,\overline{\Delta\log L}_{p,t-1}
+\gamma_{h(t)}+\delta_{d(t)}+\eta\,\text{week}(t)
$$

`base_fee` and `ewma_vol` enter at `t`; every other regressor is lagged one block to be
predetermined. $\alpha_p$ are pool-pair fixed effects. All logic lives in `arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (4725, 28)


## Onset risk set

Rows with `gap_lag == 0`, dropping pairs with no `D` variation (uninformative under pair FE).

In [2]:
onset = est.build_risk_set(panel, quantile=0.2, condition="onset")

q20: dropping 10 pool pairs with no D variation (uninformative under pair FE): ['uniswap_2_vs_pancake_2', 'uniswap_2_vs_uniswap_3', 'uniswap_2_vs_uniswap_4', 'uniswap_3_vs_pancake_1', 'uniswap_3_vs_pancake_2', 'uniswap_3_vs_uniswap_4', 'uniswap_3_vs_uniswap_5', 'uniswap_4_vs_pancake_1', 'uniswap_4_vs_pancake_2', 'uniswap_4_vs_uniswap_5']
onset: (2216, 17) | D mean: 0.0501 | pairs: 11


## Fit — logit & probit

Pool-pair fixed effects `C(pair)`, cluster-robust SEs by pair. Covariates are mean-centred so the
intercept is read at an average observation.

In [3]:
onset_c = est.center_continuous(onset, est.ONSET_TERMS)
res_logit  = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="logit")
res_probit = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="probit")
print(res_logit.summary())
print(res_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:                 2216
Model:                          Logit   Df Residuals:                     2198
Method:                           MLE   Df Model:                           17
Date:                Wed, 05 Aug 2026   Pseudo R-squ.:                  0.1877
Time:                        13:02:16   Log-Likelihood:                -357.81
converged:                       True   LL-Null:                       -440.50
Covariance Type:              cluster   LLR p-value:                 2.310e-26
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -3.1810      0.149    -21.330      0.000      -3.473      -2.889
C(pair)[T.uniswap_1_vs_pancake_1]     0.8045      0.130      6

## Average marginal effects

Logit coefficients are not probability changes, so report AMEs (logit and probit agree closely).

In [4]:
est.average_marginal_effects(res_logit, est.ONSET_TERMS)

,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.
log_base_fee,-0.167679,0.056628,-2.961063,0.003066,-0.278668,-0.056690
gas_util_lag,-0.035701,0.015919,-2.242630,0.024921,-0.066903,-0.004500
tip_p90_lag,-0.000420,0.001742,-0.241152,0.809438,-0.003835,0.002995
mev_lag,-0.004810,0.007696,-0.624969,0.531992,-0.019894,0.010274
freq_lag,0.070602,0.015386,4.588828,0.000004,0.040447,0.100757
log_vol,0.260106,0.140980,1.844979,0.065041,-0.016211,0.536422
dlogL_lag,-0.014041,0.004680,-3.000333,0.002697,-0.023213,-0.004869


## Multicollinearity check

VIFs (worry above ~10) and the covariate correlation matrix.

In [5]:
print(est.variance_inflation(onset, est.ONSET_TERMS))
onset[est.ONSET_TERMS].corr().round(3)

const           98476.657605
log_base_fee        1.768493
gas_util_lag        1.135467
tip_p90_lag         1.024887
mev_lag             1.765243
freq_lag            1.605363
log_vol             1.746753
dlogL_lag           1.001477
Name: VIF, dtype: float64


,log_base_fee,gas_util_lag,tip_p90_lag,mev_lag,freq_lag,log_vol,dlogL_lag
log_base_fee,1.000,0.263,0.028,0.254,0.165,0.607,-0.025
gas_util_lag,0.263,1.000,0.152,0.015,0.036,0.021,-0.016
tip_p90_lag,0.028,0.152,1.000,-0.017,0.008,-0.007,0.019
mev_lag,0.254,0.015,-0.017,1.000,0.610,0.344,-0.001
freq_lag,0.165,0.036,0.008,0.610,1.000,0.167,-0.000
log_vol,0.607,0.021,-0.007,0.344,0.167,1.000,-0.002
dlogL_lag,-0.025,-0.016,0.019,-0.001,-0.000,-0.002,1.000
